[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter3/split-pdf.ipynb)

# Chapter 3: PDF Splitting for Large Documents

When working with large PDF documents in RAG systems, it's often necessary to split them into smaller chunks for processing. This is useful because:

1. **Memory constraints**: Large PDFs may exceed memory limits during parsing
2. **Parallel processing**: Smaller chunks can be processed concurrently
3. **Incremental updates**: Only re-process changed sections

This notebook demonstrates how to split a PDF into smaller files based on page count using PyPDF2.

**What you'll learn:**
- Download and open PDFs from URLs for processing
- Split large PDFs into smaller page-based chunks
- Write individual chunks as separate PDF files

**Prerequisites:** `pip install PyPDF2 requests`

In [21]:
%pip install --quiet PyPDF2 requests

Note: you may need to restart the kernel to use updated packages.


c:\genai\prodapplication\hands-on-rag\.venv\Scripts\python.exe: No module named pip


In [22]:
import io
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path, PurePosixPath
from typing import Final
from urllib.parse import urlparse

import requests
from PyPDF2 import PdfReader, PdfWriter
from PyPDF2.errors import PdfReadError

## PDF Splitting Functions

The `split_pdf` function downloads a PDF from a URL and splits it into smaller files:
- `download_pdf` downloads the PDF into memory, checks that it is a PDF, and opens it with `PdfReader`
- `calculate_chunk_count` calculates the number of chunks based on `pages_per_chunk`
- `write_pdf_chunk` writes each chunk of pages to a separate PDF file

Progress messages are printed to the cell output with `print`.

In [20]:
'''OPEN A FILE- Open a file- behind the scenes, Python has to locate your file, make sure it exists, and 
ask the operating system for access to the file—after all, you might not have 
permission to access it. we add a path to our filename, which tells 
open where to look for the file. There are actually two kinds 
of paths: a relative path or an absolute one. A relative path is a 
description of the file’s location that is relative to the folder 
(we usually say directory instead when we talk about paths) 
you ran your Python code from.  You can use the .. (two 
periods) notation to specify a directory one level higher.'''
''' READ A FILE- the 
contents of the file are taken and placed into a Python string, which you can 
examine or process as you wish using standard Python string operations.
There are actually two ways to read the contents of your file into 
a string: you can read the entire file all at once, or you can read 
the file a line at a time.
my_file = open('lib.txt', 'r')
my_text = my_file.read()
print(my_text)
my_file.close()
Each line ends with 
an actual new line 
character, otherwise 
known as a newline, 
which is represented by 
the escape character 
sequence ‘\n’.

my_file = open('lib.txt', 'r')
line1 = my_file.readline()
print(line1) 
line2 = my_file.readline()
print(line2)

OR
my_file = open('lib.txt', 'r')
for line in my_file:
    print(line)
my_file.close()
'''




REQUEST_TIMEOUT_SECONDS: Final = 30
DEFAULT_BASE_FILENAME: Final = "output"
PDF_HEADER_MARKER: Final = b"%PDF-"
# PDF readers accept a header that starts anywhere in the first 1024 bytes.
PDF_HEADER_SEARCH_LENGTH_BYTES: Final = 1024

In [19]:
class PdfSplitError(Exception):
    """Base class for failures while downloading or splitting a PDF."""


class InvalidPdfError(PdfSplitError):
    """Raised when the downloaded file is not a PDF or cannot be parsed."""


class PdfOutputError(PdfSplitError):
    """Raised when the output directory or a chunk file cannot be written."""


@dataclass(frozen=True, kw_only=True, slots=True)
class DownloadedPdf:
    """A PDF opened in memory, with the base name used for its chunk files."""

    reader: PdfReader
    base_filename: str
    total_pages: int

In [ ]:
def get_base_filename_from_url(pdf_url: str) -> str:
    """
    Returns the file name without its extension from the URL path.

    Returns DEFAULT_BASE_FILENAME when the URL path does not end in a file name
    with an extension, for example "https://example.com/download/".
    """
    parsed_url = urlparse(pdf_url)
    url_file_name = PurePosixPath(parsed_url.path).name

    if url_file_name == "":
        return DEFAULT_BASE_FILENAME

    # Basic check for a file name extension, such as "book.pdf".
    if "." not in url_file_name:
        return DEFAULT_BASE_FILENAME

    base_filename = PurePosixPath(url_file_name).stem
    return base_filename


def is_pdf_content(file_bytes: bytes) -> bool:
    """Returns True when the PDF header marker is in the first 1024 bytes."""
    header_bytes = file_bytes[:PDF_HEADER_SEARCH_LENGTH_BYTES]
    contains_pdf_header = PDF_HEADER_MARKER in header_bytes
    return contains_pdf_header


def validate_pages_per_chunk(pages_per_chunk: int) -> None:
    """
    Raises:
        ValueError: pages_per_chunk is less than 1.
    """
    if pages_per_chunk < 1:
        raise ValueError(f"pages_per_chunk must be 1 or more, got {pages_per_chunk}.")


def calculate_chunk_count(*, total_pages: int, pages_per_chunk: int) -> int:
    """
    Returns how many chunks of at most pages_per_chunk pages hold total_pages pages.

    Raises:
        ValueError: pages_per_chunk is less than 1.
    """
    validate_pages_per_chunk(pages_per_chunk)

    # Integer ceiling division: the last chunk holds the remaining pages.
    chunk_count = (total_pages + pages_per_chunk - 1) // pages_per_chunk
    return chunk_count

In [ ]:
def download_pdf(pdf_url: str) -> DownloadedPdf:
    """
    Downloads a PDF from a URL into memory and opens it with PdfReader.

    Returns:
        The PdfReader, the base filename for the chunk files, and the page count.

    Raises:
        requests.HTTPError: The server returned a 4xx or 5xx status code.
        requests.RequestException: The download failed or timed out.
        InvalidPdfError: The downloaded file is not a PDF or cannot be parsed.
    """
    response = requests.get(pdf_url, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()

    # The whole PDF is held in memory. This is acceptable for book-sized files.
    pdf_bytes = response.content

    if not is_pdf_content(pdf_bytes):
        content_type = response.headers.get("Content-Type")
        raise InvalidPdfError(
            f"The file downloaded from {pdf_url!r} is not a PDF "
            f"(Content-Type: {content_type!r})."
        )

    pdf_stream = io.BytesIO(pdf_bytes)

    try:
        reader = PdfReader(pdf_stream)
        total_pages = len(reader.pages)
    except PdfReadError as error:
        raise InvalidPdfError(
            f"Could not parse the PDF downloaded from {pdf_url!r}."
        ) from error

    base_filename = get_base_filename_from_url(pdf_url)

    print(f"Successfully downloaded and opened PDF from URL. Total pages: {total_pages}")

    downloaded_pdf = DownloadedPdf(
        reader=reader,
        base_filename=base_filename,
        total_pages=total_pages,
    )
    return downloaded_pdf


def write_pdf_chunk(
    *,
    reader: PdfReader,
    start_page_index: int,
    end_page_index: int,
    output_path: Path,
) -> None:
    """
    Writes the pages from start_page_index (inclusive) to end_page_index (exclusive)
    into a new PDF file at output_path. Overwrites output_path if it exists.

    Raises:
        PdfOutputError: The chunk file could not be written.
    """
    writer = PdfWriter()

    for page_index in range(start_page_index, end_page_index):
        page = reader.pages[page_index]
        writer.add_page(page)

    try:
        with output_path.open("wb") as output_file:
            writer.write(output_file)
    except OSError as error:
        raise PdfOutputError(
            f"Could not write PDF chunk to {str(output_path)!r}."
        ) from error

In [ ]:
def split_pdf(*, pdf_url: str, output_directory: Path, pages_per_chunk: int) -> None:
    """
    Downloads a PDF from a URL and splits it into multiple smaller PDF files.

    Args:
        pdf_url: The URL of the PDF.
        output_directory: The directory where the PDF chunks are saved.
            It is created if it does not exist.
        pages_per_chunk: The maximum number of pages in each chunk. Must be 1 or more.

    Raises:
        ValueError: pages_per_chunk is less than 1.
        requests.RequestException: The PDF could not be downloaded.
        InvalidPdfError: The downloaded file is not a valid PDF.
        PdfOutputError: The output directory or a chunk file could not be written.
    """
    # Check the argument before the download, so a bad value fails immediately.
    validate_pages_per_chunk(pages_per_chunk)

    downloaded_pdf = download_pdf(pdf_url)
    reader = downloaded_pdf.reader
    base_filename = downloaded_pdf.base_filename
    total_pages = downloaded_pdf.total_pages

    try:
        output_directory.mkdir(parents=True, exist_ok=True)
    except OSError as error:
        raise PdfOutputError(
            f"Could not create output directory {str(output_directory)!r}."
        ) from error

    print(f"Output directory '{output_directory}' ensured.")

    chunk_count = calculate_chunk_count(
        total_pages=total_pages,
        pages_per_chunk=pages_per_chunk,
    )
    print(f"Splitting into {chunk_count} chunks of max {pages_per_chunk} pages each.")

    for chunk_index in range(chunk_count):
        chunk_number = chunk_index + 1
        start_page_index = chunk_index * pages_per_chunk
        # The last chunk can have fewer pages than pages_per_chunk.
        end_page_index = min(start_page_index + pages_per_chunk, total_pages)

        first_page_number = start_page_index + 1
        last_page_number = end_page_index
        print(
            f"Processing chunk {chunk_number}/{chunk_count} "
            f"(pages {first_page_number}-{last_page_number})..."
        )

        output_file_name = f"{base_filename}_chunk_{chunk_number}.pdf"
        output_path = output_directory / output_file_name

        write_pdf_chunk(
            reader=reader,
            start_page_index=start_page_index,
            end_page_index=end_page_index,
            output_path=output_path,
        )

        print(f"Chunk {chunk_number} saved as '{output_path}'")

    print("\nPDF splitting completed successfully!")

## Tests

These tests check the helper functions that have no side effects. They do not download anything or write files.

In [ ]:
def test_get_base_filename_from_url_removes_pdf_extension() -> None:
    pdf_url = "https://example.com/books/rl-book.pdf"

    actual_base_filename = get_base_filename_from_url(pdf_url)

    expected_base_filename = "rl-book"
    assert actual_base_filename == expected_base_filename


def test_get_base_filename_from_url_ignores_query_string() -> None:
    pdf_url = "https://example.com/books/rl-book.pdf?version=2"

    actual_base_filename = get_base_filename_from_url(pdf_url)

    expected_base_filename = "rl-book"
    assert actual_base_filename == expected_base_filename


def test_get_base_filename_from_url_returns_default_for_directory_url() -> None:
    pdf_url = "https://example.com/books/"

    actual_base_filename = get_base_filename_from_url(pdf_url)

    assert actual_base_filename == DEFAULT_BASE_FILENAME


def test_get_base_filename_from_url_returns_default_without_extension() -> None:
    pdf_url = "https://example.com/download"

    actual_base_filename = get_base_filename_from_url(pdf_url)

    assert actual_base_filename == DEFAULT_BASE_FILENAME


def test_is_pdf_content_accepts_pdf_header() -> None:
    file_bytes = b"%PDF-1.7\n%binary marker\n"

    actual_result = is_pdf_content(file_bytes)

    assert actual_result is True


def test_is_pdf_content_rejects_html_page() -> None:
    file_bytes = b"<!DOCTYPE html><html><body>Not found</body></html>"

    actual_result = is_pdf_content(file_bytes)

    assert actual_result is False


def test_is_pdf_content_rejects_empty_bytes() -> None:
    file_bytes = b""

    actual_result = is_pdf_content(file_bytes)

    assert actual_result is False


def test_calculate_chunk_count_rounds_up_partial_last_chunk() -> None:
    actual_chunk_count = calculate_chunk_count(total_pages=352, pages_per_chunk=50)

    expected_chunk_count = 8
    assert actual_chunk_count == expected_chunk_count


def test_calculate_chunk_count_handles_exact_multiple() -> None:
    actual_chunk_count = calculate_chunk_count(total_pages=100, pages_per_chunk=50)

    expected_chunk_count = 2
    assert actual_chunk_count == expected_chunk_count


def test_calculate_chunk_count_returns_zero_for_zero_pages() -> None:
    actual_chunk_count = calculate_chunk_count(total_pages=0, pages_per_chunk=50)

    expected_chunk_count = 0
    assert actual_chunk_count == expected_chunk_count


def test_calculate_chunk_count_rejects_zero_pages_per_chunk() -> None:
    raised_error: ValueError | None = None

    try:
        calculate_chunk_count(total_pages=10, pages_per_chunk=0)
    except ValueError as error:
        raised_error = error

    assert raised_error is not None
    assert "pages_per_chunk must be 1 or more" in str(raised_error)


notebook_tests: list[Callable[[], None]] = [
    test_get_base_filename_from_url_removes_pdf_extension,
    test_get_base_filename_from_url_ignores_query_string,
    test_get_base_filename_from_url_returns_default_for_directory_url,
    test_get_base_filename_from_url_returns_default_without_extension,
    test_is_pdf_content_accepts_pdf_header,
    test_is_pdf_content_rejects_html_page,
    test_is_pdf_content_rejects_empty_bytes,
    test_calculate_chunk_count_rounds_up_partial_last_chunk,
    test_calculate_chunk_count_handles_exact_multiple,
    test_calculate_chunk_count_returns_zero_for_zero_pages,
    test_calculate_chunk_count_rejects_zero_pages_per_chunk,
]

for notebook_test in notebook_tests:
    # A failing assert stops the cell with an AssertionError at the failing line.
    notebook_test()
    test_name = notebook_test.__name__
    print(f"Passed: {test_name}")

Let's split the Sutton & Barto Reinforcement Learning textbook (352 pages) into 50-page chunks.

In [ ]:
book_pdf_url = (
    "https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf"
)
chunk_output_directory = Path("pdf_chunks")
book_pages_per_chunk = 50

split_pdf(
    pdf_url=book_pdf_url,
    output_directory=chunk_output_directory,
    pages_per_chunk=book_pages_per_chunk,
)